# May–June inflow streamflow — final 3-reservoir set

Trimmed to match the multi-watershed extraction (`Multi_Water_Shed_Extraction`)
and the storage-modeling pipeline, which settled on **three** reservoirs — the
largest in three distinct Colorado basins — once the analysis window was
extended back to 1980:

| Reservoir | Basin | Inflow gauge (this notebook) |
|---|---|---|
| Navajo | San Juan | `09346400` San Juan R **nr Carracas, CO** |
| Blue Mesa | Gunnison | `09114500` Gunnison R **at Gunnison, CO** |
| Pueblo | Arkansas | `07097000` Arkansas R **at Portland, CO** |

Dropped from the earlier 10-gauge run: John Martin, Lake Granby, McPhee,
Dillon, Green Mountain, Twin Lakes, Vallecito.

### Why these gauge IDs differ from the watershed-extraction gauge IDs
The watershed notebook delineates each reservoir's **full drainage area**, so it
snaps to an **outlet / pour-point** gauge — below-dam for Navajo (`09355500`,
Archuleta) and Pueblo (`07099400`, above Pueblo city / below Pueblo Dam), and the
reservoir **impoundment** site for Blue Mesa (`09124600`, near Sapinero).

This notebook needs the **inflow** (snowmelt) signal *above* each reservoir, so it
uses upstream stream gauges instead. Copying the watershed gauge IDs here would
break the analysis: `09124600` is a lake/impoundment site that serves reservoir
elevation & storage — **not** `00060` discharge — so the pull returns nothing, and
`09355500` / `07099400` are below-dam **releases** (operational output), not the
natural May–June inflow the regression treats as the climate predictor.

**Window:** extended to **1980** to line up with the 1980-extended modeling window
(one-line revert to `1985` via `START_DATE` below).

In [1]:
!pip install dataretrieval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.8/164.8 kB 3.6 MB/s eta 0:00:00


In [2]:
import pandas as pd
from dataretrieval import nwis

In [3]:
# Analysis window. Extended back to 1980 to match the storage-modeling window
# that drove the 3-reservoir selection. Set START_DATE = "1985-01-01" to revert
# to the original 1985 start.
START_DATE = "1980-01-01"
END_DATE   = "2025-12-31"

# Final 3-reservoir set (Navajo, Blue Mesa, Pueblo) -- INFLOW gauges upstream of
# each reservoir, i.e. the snowmelt signal feeding storage. These intentionally
# differ from the watershed-extraction pour-point gauges (see notebook intro):
#   Navajo    09346400  San Juan River near Carracas, CO   (above Navajo Reservoir)
#   Blue Mesa 09114500  Gunnison River at Gunnison, CO      (above Blue Mesa Reservoir)
#   Pueblo    07097000  Arkansas River at Portland, CO      (above Pueblo Reservoir)
gauges = pd.DataFrame({
    "reservoir": [
        "Navajo",
        "Blue Mesa",
        "Pueblo",
    ],
    "usgs_site": [
        "09346400",
        "09114500",
        "07097000",
    ]
})
gauges

,reservoir,usgs_site
0,Navajo,09346400
1,Blue Mesa,09114500
2,Pueblo,07097000


In [5]:
def get_may_june_streamflow(site, reservoir_name, start=START_DATE, end=END_DATE):
    try:
        flow = nwis.get_record(
            sites=site,
            service="dv",
            start=start,
            end=end,
            parameterCd="00060"
        )

        if flow.empty:
            print(f"No data returned for {reservoir_name} ({site})")
            return None

        # If datetime is the index, use it
        flow = flow.reset_index()

        # Find date column
        date_col = "datetime" if "datetime" in flow.columns else flow.columns[0]

        # Find discharge column
        discharge_cols = [col for col in flow.columns if "00060" in col and "cd" not in col.lower()]
        if len(discharge_cols) == 0:
            print(f"No discharge column found for {reservoir_name} ({site})")
            print(flow.columns)
            return None

        discharge_col = discharge_cols[0]

        flow[date_col] = pd.to_datetime(flow[date_col])
        flow["year"] = flow[date_col].dt.year
        flow["month"] = flow[date_col].dt.month

        may_june = flow[flow["month"].isin([5, 6])].copy()

        annual = (
            may_june
            .groupby("year")[discharge_col]
            .mean()
            .reset_index()
            .rename(columns={discharge_col: "may_june_mean_cfs"})
        )

        annual["reservoir"] = reservoir_name
        annual["usgs_site"] = site

        return annual[["reservoir", "usgs_site", "year", "may_june_mean_cfs"]]

    except Exception as e:
        print(f"Error for {reservoir_name} ({site}): {e}")
        return None

In [6]:
all_results = []

for _, row in gauges.iterrows():
    print(f"Downloading {row['reservoir']} — {row['usgs_site']}")

    result = get_may_june_streamflow(
        site=row["usgs_site"],
        reservoir_name=row["reservoir"]
    )

    if result is not None:
        all_results.append(result)

streamflow_all = pd.concat(all_results, ignore_index=True)

display(streamflow_all.head())
display(streamflow_all.tail())

/tmp/ipykernel_1988/2989721797.py:3: DeprecationWarning: `nwis.get_record` is deprecated and will be removed from `dataretrieval` on or after 2027-05-06; use the appropriate `waterdata.get_*()` for the service you need instead.
  flow = nwis.get_record(


/tmp/ipykernel_1988/2989721797.py:3: DeprecationWarning: `nwis.get_record` is deprecated and will be removed from `dataretrieval` on or after 2027-05-06; use the appropriate `waterdata.get_*()` for the service you need instead.
  flow = nwis.get_record(


/tmp/ipykernel_1988/2989721797.py:3: DeprecationWarning: `nwis.get_record` is deprecated and will be removed from `dataretrieval` on or after 2027-05-06; use the appropriate `waterdata.get_*()` for the service you need instead.
  flow = nwis.get_record(


,reservoir,usgs_site,year,may_june_mean_cfs
0,Navajo,09346400,1980,2430.983607
1,Navajo,09346400,1981,823.639344
2,Navajo,09346400,1982,1695.786885
3,Navajo,09346400,1983,2200.475410
4,Navajo,09346400,1984,2210.081967


,reservoir,usgs_site,year,may_june_mean_cfs
115,Pueblo,07097000,2003,1322.311475
116,Pueblo,07097000,2004,948.557377
117,Pueblo,07097000,2005,1291.344262
118,Pueblo,07097000,2006,1343.180328
119,Pueblo,07097000,2007,1980.163934


In [7]:
# Filename reflects the 3-reservoir set and the window actually used.
out_name = f"three_reservoirs_may_june_streamflow_{START_DATE[:4]}_{END_DATE[:4]}.csv"
streamflow_all.to_csv(out_name, index=False)
print("Wrote", out_name)

Wrote three_reservoirs_may_june_streamflow_1980_2025.csv


In [8]:
from google.colab import files
files.download(out_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
summary = (
    streamflow_all
    .groupby(["reservoir", "usgs_site"])
    .agg(
        first_year=("year", "min"),
        last_year=("year", "max"),
        n_years=("year", "count")
    )
    .reset_index()
)

display(summary)

,reservoir,usgs_site,first_year,last_year,n_years
0,Blue Mesa,09114500,1980,2025,46
1,Navajo,09346400,1980,2025,46
2,Pueblo,07097000,1980,2007,28
